# Sentinel-2 Full Extraction — 50 Batches

Run the cells in order. The notebook loads the existing FIRMS + OSM + WorldCover CSV, prepares Sentinel-2, validates one event, then submits 50 Earth Engine Drive exports.

In [ ]:
!pip -q install earthengine-api pandas

import math
import pandas as pd
import ee

print("Imports ready.")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("Earth Engine ready.")


In [ ]:
# Change only this path if the CSV is stored elsewhere in Drive.
INPUT_CSV = "/content/drive/MyDrive/thermal_events_with_osm_worldcover.csv"

df = pd.read_csv(INPUT_CSV)
df["start_date"] = pd.to_datetime(df["start_date"]).dt.strftime("%Y-%m-%d")

print("Shape:", df.shape)
print("Unique event IDs:", df["event_id"].nunique())
print("Columns:", len(df.columns))
print("Date range:", df["start_date"].min(), "to", df["start_date"].max())

assert len(df) == df["event_id"].nunique(), "Duplicate event IDs found."
assert df["latitude"].notna().all()
assert df["longitude"].notna().all()
assert df["start_date"].notna().all()

print("Input validation passed.")


In [ ]:
S2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
S2_CLOUDS = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")

join_filter = ee.Filter.equals(
    leftField="system:index",
    rightField="system:index"
)

S2_JOINED = ee.ImageCollection(
    ee.Join.inner().apply(
        primary=S2,
        secondary=S2_CLOUDS,
        condition=join_filter
    )
).map(
    lambda pair: ee.Image(pair.get("primary")).set(
        "cloud_probability",
        ee.Image(pair.get("secondary"))
    )
)

print("S2 images:", S2.size().getInfo())
print("Joined S2 images:", S2_JOINED.size().getInfo())


In [ ]:
CLOUD_THRESHOLD = 40
RADIUS_METERS = 100
SCALE_METERS = 60

def prepare_s2_raw(image):
    image = ee.Image(image)
    cloud_prob = ee.Image(
        image.get("cloud_probability")
    ).select("probability")

    clear_mask = cloud_prob.lt(CLOUD_THRESHOLD)

    bands = image.select([
        "B2", "B3", "B4", "B8", "B11", "B12"
    ]).multiply(0.0001)

    return bands.updateMask(clear_mask)


S2_FALLBACK = (
    ee.Image.constant([0, 0, 0, 0, 0, 0])
    .rename(["B2", "B3", "B4", "B8", "B11", "B12"])
    .updateMask(ee.Image.constant(0))
)

S2_FALLBACK_PREPARED = S2_FALLBACK

def extract_event_features(feature):
    feature = ee.Feature(feature)
    point = feature.geometry()
    event_date = ee.Date(feature.get("start_date"))

    candidates = (
        S2_JOINED
        .filterBounds(point)
        .filterDate(event_date.advance(-30, "day"), event_date)
        .filter(ee.Filter.lte(
            "CLOUDY_PIXEL_PERCENTAGE",
            CLOUD_THRESHOLD
        ))
        .sort("system:time_start", False)
    )

    candidate_count = candidates.size()
    selected_image = ee.Image(candidates.first())

    real_prepared = prepare_s2_raw(selected_image)

    prepared = ee.Image(
        ee.Algorithms.If(
            candidate_count.gt(0),
            real_prepared,
            S2_FALLBACK_PREPARED
        )
    )

    stats = prepared.reduceRegion(
        reducer=ee.Reducer.mean().combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        ),
        geometry=point.buffer(RADIUS_METERS),
        scale=SCALE_METERS,
        maxPixels=1000000,
        bestEffort=True
    )

    return feature.set(stats).set({
        "S2_candidate_count": candidate_count,
        "S2_image_id": ee.Algorithms.If(
            candidate_count.gt(0),
            selected_image.get("system:index"),
            None
        ),
        "S2_image_date": ee.Algorithms.If(
            candidate_count.gt(0),
            ee.Date(
                selected_image.get("system:time_start")
            ).format("YYYY-MM-dd"),
            None
        ),
        "S2_scene_cloud": ee.Algorithms.If(
            candidate_count.gt(0),
            selected_image.get("CLOUDY_PIXEL_PERCENTAGE"),
            None
        ),
        "S2_no_image": candidate_count.eq(0)
    })

print("Extraction function ready.")


In [ ]:
# One-event validation

row = df.iloc[0]

test_feature = ee.Feature(
    ee.Geometry.Point([
        float(row.longitude),
        float(row.latitude)
    ]),
    row.to_dict()
)

test_result = ee.FeatureCollection([test_feature]).map(
    extract_event_features
)

test_info = test_result.first().getInfo()

print("Test event ID:", test_info["properties"].get("event_id"))
print("S2 candidate count:", test_info["properties"].get("S2_candidate_count"))
print("S2 image date:", test_info["properties"].get("S2_image_date"))
print("Validation complete.")


In [ ]:
# Submit 50 full-data Earth Engine exports

N_BATCHES = 50
TOTAL_EVENTS = len(df)
BATCH_SIZE = math.ceil(TOTAL_EVENTS / N_BATCHES)

print("Total events:", TOTAL_EVENTS)
print("Batch size:", BATCH_SIZE)

features = []

for _, row in df.iterrows():
    properties = row.to_dict()
    properties = {
        k: (
            None if pd.isna(v)
            else v.item() if hasattr(v, "item")
            else v
        )
        for k, v in properties.items()
    }

    features.append(
        ee.Feature(
            ee.Geometry.Point([
                float(row.longitude),
                float(row.latitude)
            ]),
            properties
        )
    )

print("EE features created:", len(features))

for batch_num in range(N_BATCHES):
    start = batch_num * BATCH_SIZE
    end = min((batch_num + 1) * BATCH_SIZE, TOTAL_EVENTS)

    batch_fc = ee.FeatureCollection(features[start:end])
    batch_s2 = batch_fc.map(extract_event_features)

    task = ee.batch.Export.table.toDrive(
        collection=batch_s2,
        description=f"S2_FULL_BATCH_{batch_num+1:02d}",
        folder="S2_FULL_BATCHES",
        fileNamePrefix=f"S2_FULL_BATCH_{batch_num+1:02d}",
        fileFormat="CSV"
    )

    task.start()

    print(
        f"Submitted batch {batch_num+1:02d}/{N_BATCHES} "
        f"events {start}:{end}"
    )

print("\nALL 50 EXPORT TASKS HAVE BEEN SUBMITTED.")
print("Drive folder: S2_FULL_BATCHES")
